In [1]:
import csv
import pandas as pd
import os
import numpy as np
from datetime import datetime

In [2]:
base_dir = os.getcwd()
idb_dir = os.path.join(base_dir, "IDB")
update_dir = os.path.join(idb_dir, "Update_2024")
output_dir = os.path.join(update_dir,"Output")

In [3]:
country_mapping = {
    'Barbados':'BRB',
    'Bahamas':'BHS',
    'Bolivia':'BOL',
    'Belize':'BLZ',
    'Ecuador':'ECU',
    'Guatemala':'GTM',
    'Guyana':'GUY',
    'Honduras':'HND',
    'Haiti':'HTI',
    'Jamaica':'JAM',
    'Peru':'PER',
    'Nicaragua':'NIC',
    'Panama':'PAN',
    'Paraguay':'PRY',
    'Suriname':'SUR',
    'El Salvador':'SLV',
    'Trinidad and Tobago':'TTO',
    'Uruguay':'URY',
    'Dominican Republic':'DOM'
}

idb = pd.read_excel(os.path.join(update_dir,"./Data/IDB_Agrimonitor_-_PSE_Agricultural_Policy_Monitoring_System__data_11182024_Cat_1-3.xlsx"), 
                    sheet_name='IDB_Agrimonitor_-_PSE_Agricultu')

In [4]:
idb_pmnt = idb[['country', 'code', 'comm_id', 'unit', 'year','Category','description','value','Commoditie']]

idb_pmnt.rename(columns={'country': 'Country_Label', 'Commoditie': 'Commodity_Label', 'comm_id':'Commodity_Code',
                         'unit':'Unit', 'year':'Year', 'value':'Value', 'code':'Code','description':'Description'}, inplace=True)


countries = ['BRAZIL','CANADA','EUROPEAN UNION','COSTA RICA','COLOMBIA','CHILE','MEXICO','UNITED STATES','ARGENTINA']
idb_pmnt = idb_pmnt[~idb_pmnt['Country_Label'].isin(countries)]
print(idb_pmnt.shape)

payments = [ 'Payments Based on Input Use',
            'Payments Based on Non-current A/AN/R/I, Production not Required',
           'Payments Based on Non-current A/AN/R/I, Production Required',
           'Payments Based on Current A/AN/R/I, Production Required',
             'Payments Based on Output', 
            'Miscellaneous Payments', 
            'Payments Based on Non-commodity Criteria', 'Variable Input Use','Fixed Capital Formation','On-farm Services']

idb_pmnt = idb_pmnt[idb_pmnt.Description.isin(payments)]  

idb_pmnt.Country_Label = idb_pmnt.Country_Label.str.lower().str.title()
idb_pmnt['Country_Label'] = np.where(idb_pmnt.Country_Label=='Trinidad And Tobago', 'Trinidad and Tobago',
                                       idb_pmnt.Country_Label)

idb_pmnt.Value = idb_pmnt.Value*10e5
idb_pmnt = idb_pmnt[idb_pmnt.Value!=0]
idb_pmnt = idb_pmnt.drop(['Unit'],axis=1)

idb_pmnt['Commodity_Label'] = np.where(idb_pmnt.Commodity_Label=='Group or not commodities', 'Group or Not Commodities',
                                       idb_pmnt.Commodity_Label)

# Panama PHR should be Crops and code 9991
idb_pmnt['Commodity_Label'] = np.where(((idb_pmnt.Country_Label=='Panama') & 
                                        (idb_pmnt.Commodity_Label=='Group or Not Commodities') &
                                        (idb_pmnt.Code=='PHR')),'Crops', idb_pmnt['Commodity_Label'])

idb_pmnt['Commodity_Code'] = np.where(((idb_pmnt.Country_Label=='Panama') 
                                        & (idb_pmnt.Code=='PHR')), 9991, idb_pmnt['Commodity_Code'])

idb_pmnt['Category'] = np.where(((idb_pmnt.Country_Label=='Panama') 
                                        & (idb_pmnt.Code=='PHR')), 3, idb_pmnt['Category'])
idb_pmnt.head()

(85476, 9)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\798228224.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  idb_pmnt.rename(columns={'country': 'Country_Label', 'Commoditie': 'Commodity_Label', 'comm_id':'Commodity_Code',


,Country_Label,Code,Commodity_Code,Year,Category,Description,Value,Commodity_Label
28,Jamaica,PIV,6,2009,3,Variable Input Use,53243452.4,Bananas
57,Jamaica,PIF,24,2007,3,Fixed Capital Formation,1471500.0,Oranges
64,Jamaica,PIS,34,2010,3,On-farm Services,709498000.0,Refined Sugar
78,Honduras,PIV,19,2011,3,Variable Input Use,27000000.0,Maize
131,Honduras,PIF,8,2012,3,Fixed Capital Formation,3378103.5,Coffee


In [5]:
ex_rate = pd.read_csv(os.path.join(update_dir,"./Data/ExchangeRates.csv"))
ex_rate.set_index('country', inplace=True)
ex_rate = pd.DataFrame(ex_rate.unstack().reset_index())
ex_rate.rename(columns={'level_0':'YEAR', 'country':'COUNTRY_LABEL', 0:'ER_OFFICIAL'}, inplace=True)
ex_rate['COUNTRY_LABEL'] = ex_rate['COUNTRY_LABEL'].str.strip()
ex_rate = ex_rate[~ex_rate['COUNTRY_LABEL'].isin(countries)]
ex_rate = ex_rate[~ex_rate['COUNTRY_LABEL'].isin(['Canada','OECD total'])]

ex_rate['EROFFICIAL_SOURCE']='IDB'
ex_rate.YEAR = ex_rate.YEAR.astype(int)
ex_rate = ex_rate[ex_rate['YEAR']>=2006]
ex_rate = ex_rate.reset_index(drop=True)
ex_rate.head()

,YEAR,COUNTRY_LABEL,ER_OFFICIAL,EROFFICIAL_SOURCE
0,2006,BAHAMAS,NaN,IDB
1,2006,BARBADOS,NaN,IDB
2,2006,BELIZE,NaN,IDB
3,2006,BOLIVIA,8.011534,IDB
4,2006,DOMINICAN REPUBLIC,33.096400,IDB


In [6]:
idb_pivot = idb_pmnt.pivot_table(index=['Country_Label','Commodity_Label','Commodity_Code',
                                               'Description','Code','Year'], columns='Category', 
                    values=['Value'])
idb_pivot = idb_pivot.sort_index(axis=1, level=1)

idb_pivot.columns = ['Cat_'+f'{y}' for x,y in idb_pivot.columns]
idb_pivot = idb_pivot.reset_index()
idb_pivot = idb_pivot.sort_values(['Country_Label','Commodity_Label','Commodity_Code','Description','Year'])
idb_pivot.head()

,Country_Label,Commodity_Label,Commodity_Code,Description,Code,Year,Cat_1,Cat_2,Cat_3
0,Bahamas,Group or Not Commodities,0,Fixed Capital Formation,PIF,2011,5000.0,NaN,NaN
1,Bahamas,Group or Not Commodities,0,Fixed Capital Formation,PIF,2012,5000.0,NaN,NaN
2,Bahamas,Group or Not Commodities,0,Fixed Capital Formation,PIF,2013,1004250.0,NaN,NaN
3,Bahamas,Group or Not Commodities,0,Fixed Capital Formation,PIF,2014,2250.0,NaN,NaN
4,Bahamas,Group or Not Commodities,0,On-farm Services,PIS,2011,20000.0,NaN,NaN


In [7]:
# Group or not commodities only exist in Category 1
idb_total = idb_pivot[idb_pivot.Commodity_Label=='Group or Not Commodities'].drop(['Cat_2','Cat_3'], axis=1)
idb_total.rename(columns={'Cat_1':'Value'}, inplace=True)
idb_total.head()

,Country_Label,Commodity_Label,Commodity_Code,Description,Code,Year,Value
0,Bahamas,Group or Not Commodities,0,Fixed Capital Formation,PIF,2011,5000.0
1,Bahamas,Group or Not Commodities,0,Fixed Capital Formation,PIF,2012,5000.0
2,Bahamas,Group or Not Commodities,0,Fixed Capital Formation,PIF,2013,1004250.0
3,Bahamas,Group or Not Commodities,0,Fixed Capital Formation,PIF,2014,2250.0
4,Bahamas,Group or Not Commodities,0,On-farm Services,PIS,2011,20000.0


In [8]:
# Individual commodity payments exists either in category 2 or 3 but not in 1
idb_comm = idb_pivot[idb_pivot.Commodity_Label!='Group or Not Commodities'].drop(['Cat_1'], axis=1)
idb_comm.head()

,Country_Label,Commodity_Label,Commodity_Code,Description,Code,Year,Cat_2,Cat_3
20,Bahamas,Poultry Meat,31,Payments Based on Input Use,PI,2010,NaN,410369.0
21,Bahamas,Poultry Meat,31,Payments Based on Input Use,PI,2011,NaN,500000.0
22,Bahamas,Poultry Meat,31,Payments Based on Input Use,PI,2012,NaN,500000.0
23,Bahamas,Poultry Meat,31,Payments Based on Input Use,PI,2013,NaN,425000.0
24,Bahamas,Poultry Meat,31,Payments Based on Input Use,PI,2014,NaN,425000.0


In [9]:
idb_comm.Commodity_Label.unique()

array(['Poultry Meat', 'Refined Sugar', 'Bananas', 'Beef and Veal',
       'Oranges', 'Pigmeat', 'Rice', 'Beans', 'Eggs', 'Maize', 'Milk',
       'Plantains', 'Potatoes', 'Quinoa', 'Soybeans', 'Cocoa Beans',
       'Coffee', 'Sorghum', 'Palm Oil', 'Pineapples', 'Sweet Potatoes',
       'Yam', 'Crops', 'Avocados', 'Sheep Meat', 'Honey', 'Peppers',
       'Apples', 'Barley', 'Mandarins', 'Wheat'], dtype=object)

In [10]:
PHR = idb_comm[idb_comm.Code=='PHR']
PI = idb_comm[idb_comm.Code=='PI']
PO = idb_comm[idb_comm.Code=='PO']
PC = idb_comm[idb_comm.Code=='PC']
PIV = idb_comm[idb_comm.Code=='PIV']
PIF = idb_comm[idb_comm.Code=='PIF']
PIS = idb_comm[idb_comm.Code=='PIS']

In [11]:
PIV_r = PIV[['Country_Label','Commodity_Label','Commodity_Code','Description','Code','Year','Cat_3']]
PIV_r.rename(columns={'Cat_3':'Value'}, inplace=True)

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\2220425038.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PIV_r.rename(columns={'Cat_3':'Value'}, inplace=True)


In [12]:
PIF_r = PIF[['Country_Label','Commodity_Label','Commodity_Code','Description','Code','Year','Cat_3']]
PIF_r.rename(columns={'Cat_3':'Value'}, inplace=True)

# PIF_r = PIF_r[~((PIF_r.Country_Label=='Belize') & (PIF_r.Year==2011))]
PIF_r[PIF_r['Country_Label']=='Belize']

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\3652908707.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PIF_r.rename(columns={'Cat_3':'Value'}, inplace=True)


,Country_Label,Commodity_Label,Commodity_Code,Description,Code,Year,Value
65,Belize,Bananas,6,Fixed Capital Formation,PIF,2011,4315522.0
66,Belize,Bananas,6,Fixed Capital Formation,PIF,2012,14317832.0
67,Belize,Bananas,6,Fixed Capital Formation,PIF,2013,29029253.0
68,Belize,Bananas,6,Fixed Capital Formation,PIF,2014,3121250.0
69,Belize,Bananas,6,Fixed Capital Formation,PIF,2015,2166446.0
70,Belize,Bananas,6,Fixed Capital Formation,PIF,2016,10561167.0
71,Belize,Bananas,6,Fixed Capital Formation,PIF,2017,9104338.0
72,Belize,Bananas,6,Fixed Capital Formation,PIF,2018,9805057.0
73,Belize,Bananas,6,Fixed Capital Formation,PIF,2019,4192904.0
74,Belize,Bananas,6,Fixed Capital Formation,PIF,2020,1372211.0


In [13]:
PIS_r = PIS[['Country_Label','Commodity_Label','Commodity_Code','Description','Code','Year','Cat_3']]
PIS_r.rename(columns={'Cat_3':'Value'}, inplace=True)

# PIS_r = PIS_r[~((PIS_r.Country_Label=='Belize') & (PIS_r.Year==2011))]
PIS_r[PIS_r['Country_Label']=='Belize']

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\2961291003.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PIS_r.rename(columns={'Cat_3':'Value'}, inplace=True)


,Country_Label,Commodity_Label,Commodity_Code,Description,Code,Year,Value
87,Belize,Beef and Veal,4,On-farm Services,PIS,2011,2681477.0
88,Belize,Beef and Veal,4,On-farm Services,PIS,2012,303230.0
89,Belize,Beef and Veal,4,On-farm Services,PIS,2013,1598691.0
90,Belize,Beef and Veal,4,On-farm Services,PIS,2014,928618.0
91,Belize,Beef and Veal,4,On-farm Services,PIS,2016,49818.0
92,Belize,Beef and Veal,4,On-farm Services,PIS,2017,49298.0
93,Belize,Beef and Veal,4,On-farm Services,PIS,2018,46584.0
94,Belize,Beef and Veal,4,On-farm Services,PIS,2019,59726.0
95,Belize,Beef and Veal,4,On-farm Services,PIS,2020,30907.0
96,Belize,Beef and Veal,4,On-farm Services,PIS,2021,29285.0


In [14]:
# Payments based on output
print(PO.shape)
PO_allexist = PO[(PO.Cat_2.notnull()) & (PO.Cat_3.notnull())]
print(PO_allexist.shape)
PO_allexist['Value'] = np.where((PO_allexist.Cat_2==PO_allexist.Cat_3), PO_allexist.Cat_3, np.nan)
PO_allexist = PO_allexist.drop(['Cat_2','Cat_3'], axis=1)


PO_cat3 = PO[(PO.Cat_2.isnull()) & (PO.Cat_3.notnull())]
PO_cat3.drop(['Cat_2'],axis=1, inplace=True)
PO_cat3.rename(columns={'Cat_3':'Value'}, inplace=True)

PO_cat2 = PO[(PO.Cat_2.notnull()) & (PO.Cat_3.isnull())]
PO_cat2.drop(['Cat_3'],axis=1, inplace=True)
PO_cat2.rename(columns={'Cat_2':'Value'}, inplace=True)

PO_r = pd.concat([PO_allexist, PO_cat2, PO_cat3])

PO_r.head()

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\3046170687.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PO_allexist['Value'] = np.where((PO_allexist.Cat_2==PO_allexist.Cat_3), PO_allexist.Cat_3, np.nan)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\3046170687.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PO_cat3.drop(['Cat_2'],axis=1, inplace=True)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\3046170687.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentatio

(109, 8)
(78, 8)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\3046170687.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PO_cat2.rename(columns={'Cat_2':'Value'}, inplace=True)


,Country_Label,Commodity_Label,Commodity_Code,Description,Code,Year,Value
616,Dominican Republic,Cocoa Beans,52,Payments Based on Output,PO,2015,1.508702e+07
617,Dominican Republic,Cocoa Beans,52,Payments Based on Output,PO,2016,1.270821e+07
618,Dominican Republic,Cocoa Beans,52,Payments Based on Output,PO,2017,1.699786e+07
619,Dominican Republic,Cocoa Beans,52,Payments Based on Output,PO,2018,1.943909e+07
620,Dominican Republic,Cocoa Beans,52,Payments Based on Output,PO,2019,3.369372e+08


In [15]:
# Payments based on input 
print(PI.shape)
# no data exists in both category for PI
PI_allexist = PI[(PI.Cat_2.notnull()) & (PI.Cat_3.notnull())]
print(PI_allexist.shape)

# In PI, value exists only in category 3, so dropping this category 
PI_cat2_exist = PI[PI.Cat_2.notnull()]
print(PI_cat2_exist.shape)
PI_r = PI[(PI.Cat_2.isnull()) & (PI.Cat_3.notnull())]
PI_r.drop(['Cat_2'],axis=1, inplace=True)
PI_r.rename(columns={'Cat_3':'Value'}, inplace=True)

# Drop Belizer, PI, 2011
# PI_r = PI_r[~((PI_r.Country_Label=='Belize') & (PI_r.Year==2011))]
print(PI_r.shape)
PI_r[PI_r['Country_Label']=='Belize']

(604, 8)
(0, 8)
(0, 8)
(604, 7)


,Country_Label,Commodity_Label,Commodity_Code,Description,Code,Year,Value
76,Belize,Bananas,6,Payments Based on Input Use,PI,2011,4315522.0
77,Belize,Bananas,6,Payments Based on Input Use,PI,2012,14317832.0
78,Belize,Bananas,6,Payments Based on Input Use,PI,2013,29029253.0
79,Belize,Bananas,6,Payments Based on Input Use,PI,2014,3121250.0
80,Belize,Bananas,6,Payments Based on Input Use,PI,2015,2166446.0
81,Belize,Bananas,6,Payments Based on Input Use,PI,2016,10561167.0
82,Belize,Bananas,6,Payments Based on Input Use,PI,2017,9104338.0
83,Belize,Bananas,6,Payments Based on Input Use,PI,2018,9805057.0
84,Belize,Bananas,6,Payments Based on Input Use,PI,2019,4192904.0
85,Belize,Bananas,6,Payments Based on Input Use,PI,2020,1372211.0


In [16]:
allB_bycom = pd.concat([PI_r, PIV_r, PIF_r, PIS_r])

allB_bycom = allB_bycom.pivot_table(index=['Country_Label','Commodity_Label','Commodity_Code',
                                               'Year'], columns='Code', values=['Value'])
allB_bycom = allB_bycom.sort_index(axis=1, level=1)

allB_bycom.columns = [f'{y}' for x,y in allB_bycom.columns]
allB_bycom = allB_bycom.reset_index()
allB_bycom.rename(columns={'PI':'B', 'PIV':'B1','PIF':'B2','PIS':'B3'}, inplace=True)

allB_bycom['B_sum'] = allB_bycom['B1'].fillna(0) + allB_bycom['B2'].fillna(0)+allB_bycom['B3'].fillna(0)
allB_bycom['Ratio'] = allB_bycom['B_sum']/allB_bycom['B']

allB_bycom = allB_bycom[['Country_Label','Commodity_Label','Commodity_Code','Year','B','B1','B2','B3','B_sum','Ratio']]

# allB_bycom.to_csv(os.path.join(update_dir,'allB_bycom.csv'))

In [17]:
allB_sum = allB_bycom[['Country_Label','Year','B','B1','B2','B3']]
allB_sum = allB_sum.groupby(['Country_Label','Year']).sum()[['B','B1','B2','B3']].reset_index()
allB_sum.rename(columns={'B':'B_sum','B1':'B1_sum','B2':'B2_sum','B3':'B3_sum'}, inplace=True)
allB_sum.head()

,Country_Label,Year,B_sum,B1_sum,B2_sum,B3_sum
0,Bahamas,2010,410369.0,410369.0,0.0,0.0
1,Bahamas,2011,500000.0,500000.0,0.0,0.0
2,Bahamas,2012,500000.0,500000.0,0.0,0.0
3,Bahamas,2013,425000.0,425000.0,0.0,0.0
4,Bahamas,2014,425000.0,425000.0,0.0,0.0


In [18]:
# Payments based on input 
print(PHR.shape)
PHR_allexist = PHR[(PHR.Cat_2.notnull()) & (PHR.Cat_3.notnull())]
print(PHR_allexist.shape)

# In PHR, value exists only in category 3, so dropping this category 
PHR_cat2_exist = PHR[PHR.Cat_2.notnull()]
print(PHR_cat2_exist.shape)
PHR_r = PHR[(PHR.Cat_2.isnull()) & (PHR.Cat_3.notnull())]
PHR_r.drop(['Cat_2'],axis=1, inplace=True)
PHR_r.rename(columns={'Cat_3':'Value'}, inplace=True)

print(PHR_r.shape)

# PHR_r.to_csv('PHR_r.csv', index=False)

(10, 8)
(0, 8)
(0, 8)
(10, 7)


In [19]:
PC_r  = PC.drop(['Cat_2'], axis=1)
PC_r.rename(columns={'Cat_3':'Value'}, inplace=True)
PC_r.head()

,Country_Label,Commodity_Label,Commodity_Code,Description,Code,Year,Value
53,Barbados,Refined Sugar,34,"Payments Based on Current A/AN/R/I, Production...",PC,2011,81629.000
54,Barbados,Refined Sugar,34,"Payments Based on Current A/AN/R/I, Production...",PC,2012,29585.750
55,Barbados,Refined Sugar,34,"Payments Based on Current A/AN/R/I, Production...",PC,2013,3041.750
56,Barbados,Refined Sugar,34,"Payments Based on Current A/AN/R/I, Production...",PC,2014,3812.250
518,Bolivia,Quinoa,105,"Payments Based on Current A/AN/R/I, Production...",PC,2013,79410.002


In [20]:
PN_PHNR_PM_r = idb_total[idb_total.Code.isin(['PN', 'PM','PHNR'])]
PN_PHNR_PM_r.Commodity_Label = PN_PHNR_PM_r.Commodity_Label.str.replace('Group or Not Commodities','Unallocated')
PN_PHNR_PM_r.Commodity_Code = 9999
print(PN_PHNR_PM_r.shape)
PN_PHNR_PM_r.head()

(85, 7)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\2456209448.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PN_PHNR_PM_r.Commodity_Label = PN_PHNR_PM_r.Commodity_Label.str.replace('Group or Not Commodities','Unallocated')
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\2456209448.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PN_PHNR_PM_r.Commodity_Code = 9999


,Country_Label,Commodity_Label,Commodity_Code,Description,Code,Year,Value
34,Barbados,Unallocated,9999,Miscellaneous Payments,PM,2012,1000000.0
35,Barbados,Unallocated,9999,Miscellaneous Payments,PM,2013,3893750.0
36,Barbados,Unallocated,9999,Miscellaneous Payments,PM,2014,726685.0
121,Belize,Unallocated,9999,Miscellaneous Payments,PM,2011,97870.0
122,Belize,Unallocated,9999,Miscellaneous Payments,PM,2012,156023.0


In [21]:
payment_appn = pd.concat([PO_r, PI_r, PIV_r, PIF_r, PIS_r, PHR_r, PN_PHNR_PM_r, PC_r])
print(payment_appn.shape)
# payment_appn.to_csv(os.path.join(update_dir,'payment_appn.csv'))

(1913, 7)


In [22]:
payment_sumup = payment_appn.groupby(['Country_Label', 'Description','Code','Year']).sum()[['Value']].reset_index()
payment_sumup.shape

(573, 5)

In [23]:
total_r = idb_total[['Country_Label','Description','Code','Year','Value']]
total_r.rename(columns={'Value':'Total'}, inplace=True)
print(total_r.shape)

# Drop Belize 2014 as the payment is negative
total_r = total_r[~((total_r.Country_Label=='Belize') & (total_r.Year==2014) & (total_r.Code=='PO'))]

print(total_r.shape)

(953, 5)
(952, 5)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\425888692.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  total_r.rename(columns={'Value':'Total'}, inplace=True)


In [24]:
total_b = total_r[['Country_Label','Code','Year','Total']][total_r.Code.isin(['PI','PIV','PIF','PIS'])]

total_b = total_b.pivot_table(index=['Country_Label', 'Year'], columns='Code', values=['Total'])
total_b = total_b.sort_index(axis=1, level=1)

total_b.columns = [f'{y}' for x,y in total_b.columns]
total_b = total_b.reset_index()
total_b.rename(columns={'PI':'B', 'PIV':'B1','PIF':'B2','PIS':'B3'}, inplace=True)

total_b['B_sum'] = total_b['B1'].fillna(0) + total_b['B2'].fillna(0)+total_b['B3'].fillna(0)
total_b['Ratio'] = total_b['B_sum']/total_b['B']

total_b = total_b[['Country_Label','Year','B','B1','B2','B3','B_sum','Ratio']]
print(total_b.shape)

# total_b.to_csv(os.path.join(update_dir, 'IDB_Total_Bs_Add_Up.csv'), index=False)

(205, 8)


In [25]:
total_bs = total_b[['Country_Label','Year','B','B1','B2','B3']]
total_bs.shape

(205, 6)

In [26]:
B_left = total_bs.merge(allB_sum, how='right')
# B_left.to_csv(os.path.join(update_dir, 'IDB_B_right.csv'), index=False)

In [27]:
payment_check_left = total_r.merge(payment_sumup, how='left')
print(payment_check_left.shape)
# payment_check_left.to_csv(os.path.join(update_dir,'IDB_payment_left.csv'), index=False)

(952, 6)


In [28]:
payment_check_right = total_r.merge(payment_sumup, how='right')
print(payment_check_right.shape)
# payment_check_right.to_csv(os.path.join(update_dir,'IDB_payment_right.csv'), index=False)

(573, 6)


In [29]:
payment_unallocated = payment_check_left[payment_check_left.Value.isnull()]
# payment_unallocated.to_csv(os.path.join(output_dir, 'IDB_No_Commodity_Specific_Payment.csv'), index=False)

payment_unallocated = payment_unallocated[['Country_Label','Description','Code','Year','Total']]
payment_unallocated.rename(columns={'Total':'Value'}, inplace=True)
payment_unallocated['Commodity_Label'] ='Unallocated'
payment_unallocated['Commodity_Code'] = 9999
payment_unallocated = payment_unallocated[['Country_Label','Commodity_Label','Commodity_Code', 
                                           'Description','Code','Year','Value']]

payment_unallocated.head()

,Country_Label,Commodity_Label,Commodity_Code,Description,Code,Year,Value
0,Bahamas,Unallocated,9999,Fixed Capital Formation,PIF,2011,5000.0
1,Bahamas,Unallocated,9999,Fixed Capital Formation,PIF,2012,5000.0
2,Bahamas,Unallocated,9999,Fixed Capital Formation,PIF,2013,1004250.0
3,Bahamas,Unallocated,9999,Fixed Capital Formation,PIF,2014,2250.0
4,Bahamas,Unallocated,9999,On-farm Services,PIS,2011,20000.0


In [30]:
payment_f = payment_appn.append(payment_unallocated).reset_index(drop=True)
payment_f['Country_Code'] = payment_f.Country_Label.map(country_mapping)
payment_f.drop(['Description'], inplace=True, axis=1)
payment_f.head()

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\989182634.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  payment_f = payment_appn.append(payment_unallocated).reset_index(drop=True)


,Country_Label,Commodity_Label,Commodity_Code,Code,Year,Value,Country_Code
0,Dominican Republic,Cocoa Beans,52,PO,2015,1.508702e+07,DOM
1,Dominican Republic,Cocoa Beans,52,PO,2016,1.270821e+07,DOM
2,Dominican Republic,Cocoa Beans,52,PO,2017,1.699786e+07,DOM
3,Dominican Republic,Cocoa Beans,52,PO,2018,1.943909e+07,DOM
4,Dominican Republic,Cocoa Beans,52,PO,2019,3.369372e+08,DOM


In [31]:
payment_sumup = payment_f.groupby(['Country_Label','Country_Code', 'Code','Year']).sum()[['Value']].reset_index()
payment_check = total_r.merge(payment_sumup, how='right')
print(payment_check.shape)
payment_check['gap'] = payment_check.Total-payment_check.Value
payment_check.head()
payment_check.to_csv(os.path.join(update_dir, 'payment_gap.csv'), index=False)

(962, 7)


In [32]:
payment_gap = payment_check[['Country_Label','Code','Year','gap','Country_Code']]
print(payment_gap.shape)
payment_gap = payment_gap[payment_gap.gap!=0]
print(payment_gap.shape)
payment_gap = payment_gap[payment_gap.gap.notnull()]
print(payment_gap.shape)
payment_gap = payment_gap[payment_gap.gap>0]
print(payment_gap.shape)

payment_gap.rename(columns={'gap':'Value'}, inplace=True)
payment_gap['Commodity_Label'] = 'Unallocated'
payment_gap['Commodity_Code'] = 9999
payment_gap = payment_gap[['Country_Label','Commodity_Label','Commodity_Code','Code','Year','Value','Country_Code']]
payment_gap.head()

(962, 5)
(437, 5)
(427, 5)
(427, 5)


,Country_Label,Commodity_Label,Commodity_Code,Code,Year,Value,Country_Code
2,Bahamas,Unallocated,9999,PI,2010,13087.0,BHS
3,Bahamas,Unallocated,9999,PI,2011,65000.0,BHS
4,Bahamas,Unallocated,9999,PI,2012,65000.0,BHS
5,Bahamas,Unallocated,9999,PI,2013,1658250.0,BHS
6,Bahamas,Unallocated,9999,PI,2014,1256250.0,BHS


In [33]:
payment_gapfilled = payment_f.append(payment_gap)
payment_gapfilled.head()
# payment_gapfilled.to_csv('payment_gapfilled.csv', index=False)

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\2082271963.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  payment_gapfilled = payment_f.append(payment_gap)


,Country_Label,Commodity_Label,Commodity_Code,Code,Year,Value,Country_Code
0,Dominican Republic,Cocoa Beans,52,PO,2015,1.508702e+07,DOM
1,Dominican Republic,Cocoa Beans,52,PO,2016,1.270821e+07,DOM
2,Dominican Republic,Cocoa Beans,52,PO,2017,1.699786e+07,DOM
3,Dominican Republic,Cocoa Beans,52,PO,2018,1.943909e+07,DOM
4,Dominican Republic,Cocoa Beans,52,PO,2019,3.369372e+08,DOM


In [34]:
payment_sumup = payment_gapfilled.groupby(['Country_Label','Country_Code', 'Code','Year']).sum()[['Value']].reset_index()
payment_check = total_r.merge(payment_sumup, how='right')
print(payment_sumup.shape)
# payment_check.to_csv(os.path.join(update_dir, 'payment_check.csv'), index=False)

payment_check.rename(columns={'Value':'Sum_up'}, inplace=True)
payment_check['Ratio (H/G)'] = payment_check['Sum_up']/payment_check['Total']
payment_check = payment_check[['Country_Label','Country_Code','Description','Code', 'Year','Total','Sum_up','Ratio (H/G)']]
# payment_check.to_csv(os.path.join(update_dir, 'payment_final_check.csv'), index=False)

(962, 5)


In [35]:
payment_final = payment_gapfilled.pivot_table(index=['Country_Label','Country_Code','Commodity_Label','Commodity_Code',
                                               'Year'], columns='Code', values=['Value'])
payment_final = payment_final.sort_index(axis=1, level=1)
print(payment_final.shape)

payment_final.columns = [f'{y}' for x,y in payment_final.columns]
payment_final = payment_final.reset_index()
payment_final = payment_final.sort_values(['Country_Label','Country_Code','Commodity_Label','Commodity_Code','Year'])

(977, 10)


In [36]:
# Trinidad and Tobago - PHR and PHNR 
tbg = payment_final[payment_final.Country_Label=='Trinidad and Tobago']
tbg['PHNR'] = np.where(tbg.Commodity_Label=='Unallocated', tbg.PHNR.fillna(0)+tbg.PHR.fillna(0), tbg.PHNR)
tbg['PHR'] = np.where(tbg.Commodity_Label=='Unallocated', np.nan, tbg.PHR)

payment_final_oth = payment_final[payment_final.Country_Label!='Trinidad and Tobago']

payment_final = payment_final_oth.append(tbg)

#drop Suriname-Rice-2015-2018 as advised by IDB
# payment_final = payment_final[~((payment_final.Country_Label=='Suriname') & (payment_final.Commodity_Label=='Rice') & 
#                     (payment_final.Year.isin([2015,2016,2017,2018])))]


payment_final.shape

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\1824295124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tbg['PHNR'] = np.where(tbg.Commodity_Label=='Unallocated', tbg.PHNR.fillna(0)+tbg.PHR.fillna(0), tbg.PHNR)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\1824295124.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tbg['PHR'] = np.where(tbg.Commodity_Label=='Unallocated', np.nan, tbg.PHR)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\1824295124.py:8: FutureWarning: The frame.append me

(977, 15)

In [37]:
payment_final.rename(columns={'PC':'C','PHNR':'E','PHR':'D','PI':'B','PIV':'B1','PIF':'B2','PIS':'B3', 
                              'PM':'G','PN':'F','PO':'A2'}, inplace=True)

payment_final = payment_final.replace({'Commodity_Label':{'Non MPS Commodities':'NonMPS from Workbook', 
                                                          'Pigmeat':'Pig meat', 'Beef and Veal':'Beef and veal',
                                                         'Cocoa Beans':'Cocoa beans', 'Poultry Meat':'Poultry meat', 
                                                         'Sweet Potatoes':'Sweet potatoes','Yam':"Yams", 
                                                         'Sheep Meat':'Sheep meat'}})

payment_final['Commodity_Type'] = np.where(payment_final.Commodity_Label=='NonMPS from Workbook','No','Yes')

nonmps = payment_final[payment_final.Commodity_Label=='NonMPS from Workbook']
nonmps['Commodity_Label'] = nonmps.Commodity_Label + ' - ' + nonmps['Country_Label']
nonmps.Commodity_Code = 'XEJAM'

not_nonmps = payment_final[payment_final.Commodity_Label!='NonMPS from Workbook']
idb_relab = not_nonmps.append(nonmps)


idb_relab['Commodity_Type'] = np.where(idb_relab.Commodity_Label.isin(['Unallocated', 'Crops']),np.nan,
                                       idb_relab['Commodity_Type'])

idb_relab['Commodity_Label'] = np.where(idb_relab.Commodity_Label=='Refined Sugar','Refined sugar',
                                       idb_relab.Commodity_Label)

idb_relab = idb_relab[['Country_Label', 'Country_Code', 'Commodity_Label', 'Commodity_Code','Commodity_Type',
       'Year','A2', 'B','B1','B2','B3', 'C','D', 'E','F','G']]

idb_relab.head()


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_31932\1453359764.py:17: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  idb_relab = not_nonmps.append(nonmps)


,Country_Label,Country_Code,Commodity_Label,Commodity_Code,Commodity_Type,Year,A2,B,B1,B2,B3,C,D,E,F,G
0,Bahamas,BHS,Poultry meat,31,Yes,2010,NaN,410369.0,410369.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bahamas,BHS,Poultry meat,31,Yes,2011,NaN,500000.0,500000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Bahamas,BHS,Poultry meat,31,Yes,2012,NaN,500000.0,500000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Bahamas,BHS,Poultry meat,31,Yes,2013,NaN,425000.0,425000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bahamas,BHS,Poultry meat,31,Yes,2014,NaN,425000.0,425000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
idb_relab.to_csv(os.path.join(output_dir, 'IDB_Payment_Data.csv'), index=False)